# 06 — Tổng hợp dev và khóa CNN
Chọn bằng số chưa làm tròn; không đọc test.

In [ ]:
from pathlib import Path
import os, sys
# Bỏ comment nếu Kaggle gắn nhiều bộ code hoặc dữ liệu.
# os.environ['CS231_PROJECT_ROOT'] = '/kaggle/input/TEN-DATASET-CODE'
# os.environ['CS231_DATA_ROOT'] = '/kaggle/input/TEN-DATASET-ANH/DATA'
# os.environ['CS231_CNN_PREVIOUS_RESULTS'] = '/kaggle/input/.../CS231_CNN/results_224'
os.environ.setdefault('MPLCONFIGDIR', '/tmp/cs231-matplotlib')
roots = [Path.cwd(), *Path.cwd().parents]
roots += [p for pattern in ('*', 'datasets/*/*') for p in Path('/kaggle/input').glob(pattern)]
if os.environ.get('CS231_PROJECT_ROOT'):
    roots = [Path(os.environ['CS231_PROJECT_ROOT'])]
projects = sorted({p.resolve() for p in roots if (p/'src/preprocessing.py').is_file()})
if len(projects) != 1:
    raise FileNotFoundError('Đặt CS231_PROJECT_ROOT trỏ tới thư mục chứa src.')
PROJECT_ROOT = projects[0]
sys.path.insert(0, str(PROJECT_ROOT))
from src.experiments.runtime import setup, restore_previous_output
DATA_ROOT, RESULTS_ROOT = setup('CNN', PROJECT_ROOT)

In [ ]:
restore_previous_output(RESULTS_ROOT, 'CNN', 'candidates/C04_DEEP_AUG/summary.json')

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from IPython.display import display, Image
from src.CNN.data import CLASSES, CANDIDATES, load_config, configure_gpu, seed_everything
configure_gpu()
seed_everything(42)

## 1. Bảng tốt nhất từng cấu hình

In [ ]:
from src.CNN.pipeline import select_and_lock
comparison, lock = select_and_lock(RESULTS_ROOT)
display(comparison)
print('Đã khóa:', lock['candidate'], '| epoch:', lock['best_epoch'])

## 2. PCA của đặc trưng học được
Vector GAP học từ pixel, không phải HSV/HOG. PCA2D chỉ để vẽ: fit train, chiếu dev; không tham gia chọn mô hình hoặc phân loại.

In [ ]:
from src.CNN.analysis import inspect_features
coordinates, selected_channels = inspect_features(DATA_ROOT, RESULTS_ROOT)
display(coordinates.head())
display(Image(filename=str(RESULTS_ROOT/'feature_analysis/CNN_EMBEDDING_PCA.png')))

## 3. Feature map
Sáu kênh có năng lượng trung bình cao nhất ở Conv đầu/cuối của một ảnh train. Thang màu riêng từng kênh; chưa gán ý nghĩa ngữ nghĩa cho kênh.

In [ ]:
display(selected_channels)
display(Image(filename=str(RESULTS_ROOT/'feature_analysis/CNN_FEATURE_MAPS.png')))

## 4. Bàn giao checkpoint đã khóa
Bước07 nạp chính best.keras được chọn; không fit lại hoặc gộp train+dev.